<a href="https://colab.research.google.com/github/dhiaamor/drip/blob/main/drip2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
DATA_DIR = os.environ.get("DEEPFASHION_DIR", "./data/DeepFashion")

Creating a sample Dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

SAMPLE_DIR = Path("./data/DeepFashion_sample")
IMG_DIR = SAMPLE_DIR / "img"
IMG_DIR.mkdir(parents=True, exist_ok=True)

# Sample items: (filename, category_id, main_rgb)
sample_items = [
    ("top_navy.jpg", 1, (25, 45, 95)),
    ("top_white.jpg", 1, (240, 240, 242)),
    ("top_red.jpg", 1, (185, 35, 45)),
    ("bottom_denim.jpg", 2, (40, 70, 120)),
    ("bottom_black.jpg", 2, (25, 25, 28)),
    ("bottom_beige.jpg", 2, (210, 195, 165)),
    ("dress_olive.jpg", 3, (95, 115, 70)),
]

records = []
rng = np.random.default_rng(42)

for fname, cat_id, rgb in sample_items:
    # Create a 128x128 garment block with subtle texture variation
    base = np.array(rgb, dtype=np.uint8)
    noise = rng.integers(-12, 13, size=(128, 128, 3), dtype=np.int16)
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    img_path = IMG_DIR / fname
    Image.fromarray(arr).save(img_path)

    records.append({
        "image_name": f"img/{fname}",
        "category_id": cat_id,
    })

df_sample = pd.DataFrame(records)
df_sample.to_csv(SAMPLE_DIR / "sample_annotations.csv", index=False)
print("Created sample files in:", SAMPLE_DIR.resolve())

Created sample files in: /content/data/DeepFashion_sample


Verify Sample Folder

In [ ]:
import os

sample_path = "./data/DeepFashion_sample"
for root, dirs, files in os.walk(sample_path):
    level = root.replace(sample_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or sample_path}/")
    subindent = "  " * (level + 1)
    for f in sorted(files)[:15]:
        print(f"{subindent}{f}")

DeepFashion_sample/
  sample_annotations.csv
  img/
    bottom_beige.jpg
    bottom_black.jpg
    bottom_denim.jpg
    dress_olive.jpg
    top_navy.jpg
    top_red.jpg
    top_white.jpg


In [ ]:
from pathlib import Path
import pandas as pd

SAMPLE_DIR = Path("./data/DeepFashion_sample")
df = pd.read_csv(SAMPLE_DIR / "sample_annotations.csv")
df["full_image_path"] = df["image_name"].apply(lambda p: str(SAMPLE_DIR / p))

print("Rows:", len(df))
print(df.head())

Rows: 7
             image_name  category_id  \
0      img/top_navy.jpg            1   
1     img/top_white.jpg            1   
2       img/top_red.jpg            1   
3  img/bottom_denim.jpg            2   
4  img/bottom_black.jpg            2   

                                full_image_path  
0      data/DeepFashion_sample/img/top_navy.jpg  
1     data/DeepFashion_sample/img/top_white.jpg  
2       data/DeepFashion_sample/img/top_red.jpg  
3  data/DeepFashion_sample/img/bottom_denim.jpg  
4  data/DeepFashion_sample/img/bottom_black.jpg  


In [ ]:
import os
os.listdir('/content/drive/MyDrive')
os.listdir('/content/drive/MyDrive/DeepFashion')




FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DeepFashion'

In [ ]:
with open('/content/drive/MyDrive/DeepFashion/Anno_coarse/list_attr_cloth.txt', 'r') as f:
    for i, line in enumerate(f, 1):
        if i <= 10:  # Print first 10 lines
            print(f"Line {i}: {line.strip()}")
        else:
            break

Line 1: 1000
Line 2: attribute_name  attribute_type
Line 3: a-line                       3
Line 4: abstract                     1
Line 5: abstract chevron             1
Line 6: abstract chevron print       1
Line 7: abstract diamond             1
Line 8: abstract floral              1
Line 9: abstract floral print        1
Line 10: abstract geo                 1


In [ ]:
pip install Pillow scikit-learn numpy


In [ ]:
import zipfile
# Set dataset path in Google Drive
dataset_path = '/content/drive/MyDrive/DeepFashion'
anno_path = os.path.join(dataset_path, 'Anno_coarse')
img_base_path = '/content/DeepFashion/Img/img'  # Updated to point to the 'img' subdirectory

# Create the extraction directory (already done during extraction)
os.makedirs('/content/DeepFashion/img', exist_ok=True)

# Define ZIP file path for img.zip
zip_path = os.path.join(dataset_path, 'Img/img.zip')

# Extract img.zip (already executed, included for completeness)
if os.path.exists(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/DeepFashion/img')
            print(f"Extracted {zip_path} to /content/DeepFashion/img")
    except Exception as e:
        print(f"Error extracting {zip_path}: {e}")
else:
    print(f"Skipping extraction of {zip_path}: File not found")

# Verify extracted contents
print("Extracted directories:", os.listdir('/content/DeepFashion/img'))


Extracted /content/drive/MyDrive/DeepFashion/Img/img.zip to /content/DeepFashion/img
Extracted directories: ['img']


In [ ]:
import os


img_base_path = '/content/DeepFashion/img/img'
print("Contents of img_base_path:", os.listdir(img_base_path) if os.path.exists(img_base_path) else "Directory not found")

Contents of img_base_path: Directory not found


In [ ]:
import pandas as pd
import os
import gc
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans
import zipfile
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Set dataset path
dataset_path = '/content/drive/MyDrive/DeepFashion'
anno_path = os.path.join(dataset_path, 'Anno_coarse')
img_base_path = '/content/DeepFashion/img/img'
output_file = '/content/drive/MyDrive/processed_deepfashion_with_palette.csv'

# Check existing progress
if os.path.exists(output_file):
    df = pd.read_csv(output_file)
    processed_rows = len(df)
    print(f"Loaded {processed_rows} processed rows from {output_file}")
else:
    processed_rows = 0
    print("No previous progress found, starting fresh.")

# Calculate starting point
chunk_size = 1000
start_row = processed_rows
start_chunk = int(start_row / chunk_size) + 1
print(f"Resuming from chunk {start_chunk} (row {start_row})")

# File paths
category_file = os.path.join(anno_path, 'list_category_img.txt')
cloth_category_file = os.path.join(anno_path, 'list_category_cloth.txt')
attribute_file = os.path.join(anno_path, 'list_attr_img.txt')
attribute_list_file = os.path.join(anno_path, 'list_attr_cloth.txt')

# Check files
print("Checking files at paths:")
print("category_file:", category_file)
print("cloth_category_file:", cloth_category_file)
print("attribute_file:", attribute_file)
print("attribute_list_file:", attribute_list_file)

for file_path in [category_file, cloth_category_file, attribute_file, attribute_list_file]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Required file not found: {file_path}")

# Load mappings
cloth_categories = pd.read_csv(cloth_category_file, skiprows=2, sep='\s+', names=['category', 'type'])
category_to_type = {'T-shirt': 'Top', 'Shirt': 'Top', 'Sweater': 'Top', 'Jacket': 'Top', 'Blouse': 'Top',
                   'Tank': 'Top', 'Hoodie': 'Top', 'Cardigan': 'Top', 'Pants': 'Bottom', 'Jeans': 'Bottom',
                   'Shorts': 'Bottom', 'Skirt': 'Bottom', 'Leggings': 'Bottom', 'Trousers': 'Bottom',
                   'Sneakers': 'Footwear', 'Boots': 'Footwear', 'Sandals': 'Footwear', 'Shoes': 'Footwear',
                   'Heels': 'Footwear', 'Flip-Flops': 'Footwear'}
type_mapping = {1: 'Top', 2: 'Bottom', 3: 'Top'}

def parse_attribute_file(file_path):
    data = []
    with open(file_path, 'r') as f:
        next(f); next(f)
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attribute = ' '.join(parts[:-1])
                try: data.append([attribute, int(parts[-1])])
                except ValueError: print(f"Skipping line: {line.strip()}")
    return pd.DataFrame(data, columns=['attribute', 'type'])

attributes_list = parse_attribute_file(attribute_list_file)
attr_index_to_name = {i + 1: attr for i, attr in enumerate(attributes_list['attribute'])}

season_mapping = {'short sleeves': 'Summer', 'sleeveless': 'Summer', 'long sleeves': 'Winter',
                 'wool': 'Winter', 'cotton': 'Summer', 'light fabric': 'Spring', 'thick fabric': 'Autumn'}
occasion_mapping = {'formal': 'Formal', 'casual': 'Casual', 'sporty': 'Sport', 'party': 'Party', 'business': 'Work'}
style_mapping = {'casual': 'Casual', 'formal': 'Formal', 'sporty': 'Sport', 'street': 'Street'}

color_map = {'Red': {'rgb': (255, 0, 0), 'weight': 0.9}, 'Pink': {'rgb': (255, 182, 193), 'weight': 0.85},
             'Blue': {'rgb': (0, 0, 255), 'weight': 0.9}, 'Black': {'rgb': (0, 0, 0), 'weight': 0.95},
             'White': {'rgb': (255, 255, 255), 'weight': 0.9}, 'Green': {'rgb': (0, 128, 0), 'weight': 0.85},
             'Gray': {'rgb': (128, 128, 128), 'weight': 0.9}, 'Yellow': {'rgb': (255, 255, 0), 'weight': 0.85},
             'Purple': {'rgb': (128, 0, 128), 'weight': 0.85}, 'Orange': {'rgb': (255, 165, 0), 'weight': 0.85},
             'Brown': {'rgb': (165, 42, 42), 'weight': 0.8}, 'Beige': {'rgb': (245, 245, 220), 'weight': 0.8},
             'Gold': {'rgb': (255, 215, 0), 'weight': 0.85}, 'Silver': {'rgb': (192, 192, 192), 'weight': 0.85},
             'Navy': {'rgb': (0, 0, 128), 'weight': 0.9}, 'Teal': {'rgb': (0, 128, 128), 'weight': 0.85},
             'Maroon': {'rgb': (128, 0, 0), 'weight': 0.85}, 'Olive': {'rgb': (128, 128, 0), 'weight': 0.8},
             'Coral': {'rgb': (255, 127, 127), 'weight': 0.85}, 'Indigo': {'rgb': (75, 0, 130), 'weight': 0.85}}

def rgb_to_color_name(rgb, threshold=50):
    min_score = float('inf'); closest_color = 'Unknown'
    rgb = np.array(rgb)
    for color_name, color_data in color_map.items():
        target_rgb = np.array(color_data['rgb'])
        distance = np.sqrt(np.sum((rgb - target_rgb) ** 2))
        adjusted_score = distance / color_data['weight']
        if adjusted_score < min_score and adjusted_score < threshold:
            min_score = adjusted_score; closest_color = color_name
    return closest_color

def extract_color_palette(image_path):
    try:
        while image_path.startswith(('Img/', 'img/')): image_path = image_path[4:]
        full_path = os.path.join(img_base_path, image_path)
        if not os.path.exists(full_path): return ['Unknown', 'Unknown', 'Unknown']
        with Image.open(full_path) as img:
            img = img.convert('RGB').resize((100, 100))
            pixels = np.array(img).reshape(-1, 3)
            if len(pixels) < 5: return ['Unknown', 'Unknown', 'Unknown']
            kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
            kmeans.fit(pixels); cluster_centers = kmeans.cluster_centers_.astype(int)
            unique, counts = np.unique(kmeans.labels_, return_counts=True)
            top_indices = unique[np.argsort(counts)[::-1][:3]]
            palette = [tuple(cluster_centers[i]) for i in top_indices]
            return [rgb_to_color_name(color) for color in palette]
    except Exception: return ['Unknown', 'Unknown', 'Unknown']

def infer_season(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, season in season_mapping.items():
            if key in attr.lower():
                return season
    return 'All'

def infer_occasion(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, occasion in occasion_mapping.items():
            if key in attr.lower():
                return occasion
    return 'Casual'

def infer_style(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, style in style_mapping.items():
            if key in attr.lower():
                return style
    return 'Casual'

# Keep-alive
import threading
def keep_alive():
    while True: print("Keeping session alive...", time.ctime()); time.sleep(600)
thread = threading.Thread(target=keep_alive, daemon=True); thread.start()

# Process from last chunk
first_chunk = not os.path.exists(output_file)
category_iterator = pd.read_csv(category_file, skiprows=2 + start_row, sep='\s+', names=['image_path', 'category_id'], chunksize=chunk_size)
for i, categories_chunk in enumerate(category_iterator):
    print(f"Processing chunk {start_chunk + i} at {time.ctime()}...")
    attr_data = []
    with open(attribute_file, 'r') as f:
        lines = f.readlines()[2:]
        start_idx = (start_chunk + i - 1) * chunk_size
        end_idx = min(start_idx + chunk_size, len(lines))
        for line in lines[start_idx:end_idx]:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                image_path, attr_labels = parts
                attr_values = attr_labels.split()
                attr_indices = [idx + 1 for idx, val in enumerate(attr_values) if val == '1']
                attr_data.append([image_path, attr_indices])
            else: attr_data.append([parts[0], []])
    attributes_chunk = pd.DataFrame(attr_data, columns=['image_path', 'attr_labels'])

    chunk = categories_chunk.merge(attributes_chunk, on='image_path')
    del attributes_chunk

    chunk['category_name'] = chunk['category_id'].map(cloth_categories['category'])
    chunk['type'] = chunk['category_name'].map(category_to_type).fillna(chunk['category_id'].map(cloth_categories['type']).map(type_mapping))
    chunk['season'] = chunk['attr_labels'].apply(infer_season)
    chunk['occasion'] = chunk['attr_labels'].apply(infer_occasion)
    chunk['style'] = chunk['attr_labels'].apply(infer_style)
    chunk['name'] = chunk['image_path'].apply(lambda x: x.split('/')[-1].replace('.jpg', ''))
    chunk['color_palette'] = chunk['image_path'].apply(extract_color_palette)
    chunk['primary_color'] = chunk['color_palette'].apply(lambda x: x[0] if x[0] != 'Unknown' else 'Unknown')
    chunk = chunk[['name', 'type', 'primary_color', 'color_palette', 'style', 'season', 'occasion']].dropna(subset=['type'])

    chunk.to_csv(output_file, mode='a', header=False, index=False)
    del chunk; gc.collect()
    print(f"Chunk {start_chunk + i} completed. Memory cleaned.")

print(f"Preprocessing complete. Saved to '{output_file}'.")
processed_df = pd.read_csv(output_file, nrows=1000)
print(processed_df.head())
print(processed_df.groupby('type').size())
print(processed_df.groupby('primary_color').size())
print(processed_df.groupby('season').size())
print(processed_df.groupby('occasion').size())
print(processed_df.groupby('style').size())

Using local sample dataset at: /content/data/DeepFashion_sample
Sample preprocessing complete. Saved to 'data/DeepFashion_sample/processed_sample_with_palette.csv'.
        name      type primary_color               color_palette  style season occasion
    top_navy       Top       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual
   top_white       Top         White       [White, White, Beige] Casual    All   Casual
     top_red       Top         Brown       [Brown, Brown, Brown] Casual    All   Casual
bottom_denim    Bottom       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual
bottom_black    Bottom         Black     [Black, Black, Unknown] Casual    All   Casual
bottom_beige    Bottom        Silver    [Silver, Silver, Silver] Casual    All   Casual
 dress_olive Full-body       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual


In [ ]:
import gc
import os
import threading
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import KMeans

# --- 1. Configure Paths (Sample first, Google Drive fallback) ---
SAMPLE_DIR = Path("./data/DeepFashion_sample")
USE_SAMPLE = SAMPLE_DIR.exists() and (SAMPLE_DIR / "sample_annotations.csv").exists()

if USE_SAMPLE:
    print(f"Using local sample dataset at: {SAMPLE_DIR.resolve()}")
    dataset_path = str(SAMPLE_DIR)
    anno_path = str(SAMPLE_DIR)
    img_base_path = str(SAMPLE_DIR)
    output_file = str(SAMPLE_DIR / "processed_sample_with_palette.csv")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
    except ImportError:
        pass

    dataset_path = os.environ.get("DEEPFASHION_DIR", "/content/drive/MyDrive/DeepFashion")
    anno_path = os.path.join(dataset_path, "Anno_coarse")
    img_base_path = os.environ.get("DEEPFASHION_IMG_DIR", "/content/DeepFashion/img/img")
    output_file = os.path.join(dataset_path, "processed_deepfashion_with_palette.csv")

# --- 2. Category & Attribute Mappings ---
category_to_type = {
    "T-shirt": "Top", "Shirt": "Top", "Sweater": "Top", "Jacket": "Top",
    "Blouse": "Top", "Tank": "Top", "Hoodie": "Top", "Cardigan": "Top",
    "Pants": "Bottom", "Jeans": "Bottom", "Shorts": "Bottom", "Skirt": "Bottom",
    "Leggings": "Bottom", "Trousers": "Bottom",
    "Sneakers": "Footwear", "Boots": "Footwear", "Sandals": "Footwear",
    "Shoes": "Footwear", "Heels": "Footwear", "Flip-Flops": "Footwear",
    "Dress": "Full-body", "Jumpsuit": "Full-body", "Romper": "Full-body",
}
# DeepFashion coarse types: 1 = Upper-body, 2 = Lower-body, 3 = Full-body
type_mapping = {1: "Top", 2: "Bottom", 3: "Full-body"}

season_mapping = {
    "short sleeves": "Summer", "sleeveless": "Summer", "long sleeves": "Winter",
    "wool": "Winter", "cotton": "Summer", "light fabric": "Spring", "thick fabric": "Autumn",
}
occasion_mapping = {
    "formal": "Formal", "casual": "Casual", "sporty": "Sport",
    "party": "Party", "business": "Work",
}
style_mapping = {
    "casual": "Casual", "formal": "Formal", "sporty": "Sport", "street": "Street",
}

color_map = {
    "Red": {"rgb": (255, 0, 0), "weight": 0.9},
    "Pink": {"rgb": (255, 182, 193), "weight": 0.85},
    "Blue": {"rgb": (0, 0, 255), "weight": 0.9},
    "Black": {"rgb": (0, 0, 0), "weight": 0.95},
    "White": {"rgb": (255, 255, 255), "weight": 0.9},
    "Green": {"rgb": (0, 128, 0), "weight": 0.85},
    "Gray": {"rgb": (128, 128, 128), "weight": 0.9},
    "Yellow": {"rgb": (255, 255, 0), "weight": 0.85},
    "Purple": {"rgb": (128, 0, 128), "weight": 0.85},
    "Orange": {"rgb": (255, 165, 0), "weight": 0.85},
    "Brown": {"rgb": (165, 42, 42), "weight": 0.8},
    "Beige": {"rgb": (245, 245, 220), "weight": 0.8},
    "Gold": {"rgb": (255, 215, 0), "weight": 0.85},
    "Silver": {"rgb": (192, 192, 192), "weight": 0.85},
    "Navy": {"rgb": (0, 0, 128), "weight": 0.9},
    "Teal": {"rgb": (0, 128, 128), "weight": 0.85},
    "Maroon": {"rgb": (128, 0, 0), "weight": 0.85},
    "Olive": {"rgb": (128, 128, 0), "weight": 0.8},
    "Coral": {"rgb": (255, 127, 127), "weight": 0.85},
    "Indigo": {"rgb": (75, 0, 130), "weight": 0.85},
}

default_categories_df = pd.DataFrame([
    {"category": "T-shirt", "type": 1},
    {"category": "Jeans", "type": 2},
    {"category": "Dress", "type": 3},
])
attr_index_to_name = {}


def rgb_to_color_name(rgb, threshold=50):
    min_score = float("inf")
    closest_color = "Unknown"
    rgb_arr = np.array(rgb, dtype=float)
    for color_name, color_data in color_map.items():
        target_rgb = np.array(color_data["rgb"], dtype=float)
        distance = np.linalg.norm(rgb_arr - target_rgb)
        adjusted_score = distance / color_data["weight"]
        if adjusted_score < min_score and adjusted_score < threshold:
            min_score = adjusted_score
            closest_color = color_name
    return closest_color


def resolve_image_path(image_path):
    p = Path(str(image_path))
    if p.is_absolute() and p.exists():
        return p

    candidate_1 = Path(img_base_path) / p
    if candidate_1.exists():
        return candidate_1

    stripped = str(image_path).lstrip("/")
    while stripped.lower().startswith("img/"):
        stripped = stripped[4:]
    candidate_2 = Path(img_base_path) / stripped
    if candidate_2.exists():
        return candidate_2

    return None


def extract_color_palette(image_path):
    try:
        full_path = resolve_image_path(image_path)
        if full_path is None:
            return ["Unknown", "Unknown", "Unknown"]

        with Image.open(full_path) as img:
            img = img.convert("RGB").resize((100, 100))
            pixels = np.array(img).reshape(-1, 3)
            if len(pixels) < 5:
                return ["Unknown", "Unknown", "Unknown"]

            n_clusters = min(3, len(np.unique(pixels, axis=0)))
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            kmeans.fit(pixels)
            centers = kmeans.cluster_centers_.astype(int)
            unique, counts = np.unique(kmeans.labels_, return_counts=True)
            top_indices = unique[np.argsort(counts)[::-1]]
            palette = [rgb_to_color_name(centers[i]) for i in top_indices]
            while len(palette) < 3:
                palette.append("Unknown")
            return palette[:3]
    except Exception:
        return ["Unknown", "Unknown", "Unknown"]


def infer_from_mapping(attrs, mapping, default_val):
    if not isinstance(attrs, (list, tuple, np.ndarray)):
        return default_val
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), "")
        for key, mapped_val in mapping.items():
            if key in attr.lower():
                return mapped_val
    return default_val


def map_garment_type(cat_id, categories_df):
    if pd.isna(cat_id):
        return None
    idx = int(cat_id) - 1  # DeepFashion category_id is 1-indexed
    if 0 <= idx < len(categories_df):
        row = categories_df.iloc[idx]
        cat_name = row["category"]
        if cat_name in category_to_type:
            return category_to_type[cat_name]
        return type_mapping.get(int(row["type"]))
    return type_mapping.get(int(cat_id))


# --- 3. Load Data & Run Preprocessing ---
if USE_SAMPLE:
    raw_df = pd.read_csv(SAMPLE_DIR / "sample_annotations.csv")
    raw_df["attr_labels"] = [[] for _ in range(len(raw_df))]
    raw_df["category_name"] = raw_df["category_id"].apply(
        lambda cid: default_categories_df.iloc[int(cid) - 1]["category"]
        if 1 <= int(cid) <= len(default_categories_df)
        else "Item"
    )
    raw_df["type"] = raw_df["category_id"].apply(
        lambda cid: map_garment_type(cid, default_categories_df)
    )
    raw_df["season"] = raw_df["attr_labels"].apply(
        lambda a: infer_from_mapping(a, season_mapping, "All")
    )
    raw_df["occasion"] = raw_df["attr_labels"].apply(
        lambda a: infer_from_mapping(a, occasion_mapping, "Casual")
    )
    raw_df["style"] = raw_df["attr_labels"].apply(
        lambda a: infer_from_mapping(a, style_mapping, "Casual")
    )
    raw_df["name"] = raw_df["image_name"].apply(
        lambda x: Path(str(x)).stem
    )
    raw_df["color_palette"] = raw_df["image_name"].apply(extract_color_palette)
    raw_df["primary_color"] = raw_df["color_palette"].apply(
        lambda x: x[0] if x and x[0] != "Unknown" else "Unknown"
    )

    processed_out = raw_df[
        ["name", "type", "primary_color", "color_palette", "style", "season", "occasion"]
    ].dropna(subset=["type"])
    processed_out.to_csv(output_file, index=False)
    print(f"Sample preprocessing complete. Saved to '{output_file}'.")
    print(processed_out.to_string(index=False))
else:
    category_file = os.path.join(anno_path, "list_category_img.txt")
    cloth_category_file = os.path.join(anno_path, "list_category_cloth.txt")
    attribute_file = os.path.join(anno_path, "list_attr_img.txt")
    attribute_list_file = os.path.join(anno_path, "list_attr_cloth.txt")

    for fp in [category_file, cloth_category_file, attribute_file, attribute_list_file]:
        if not os.path.exists(fp):
            raise FileNotFoundError(f"Required file not found: {fp}")

    cloth_categories = pd.read_csv(
        cloth_category_file, skiprows=2, sep=r"\s+", names=["category", "type"]
    )

    attr_data_list = []
    with open(attribute_list_file, "r", encoding="utf-8", errors="ignore") as f:
        next(f)
        next(f)
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attr_data_list.append(" ".join(parts[:-1]))
    attr_index_to_name = {i + 1: attr for i, attr in enumerate(attr_data_list)}

    if os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        processed_rows = len(existing_df)
        del existing_df
        gc.collect()
    else:
        processed_rows = 0

    chunk_size = 1000
    start_chunk = (processed_rows // chunk_size) + 1
    print(f"Resuming from chunk {start_chunk} (row {processed_rows})...")

    def keep_alive():
        while True:
            time.sleep(600)
            print("Keeping session alive...", time.ctime())

    threading.Thread(target=keep_alive, daemon=True).start()

    with open(attribute_file, "r", encoding="utf-8", errors="ignore") as f:
        all_attr_lines = f.readlines()[2:]

    first_chunk = not os.path.exists(output_file)
    category_iterator = pd.read_csv(
        category_file,
        skiprows=2 + processed_rows,
        sep=r"\s+",
        names=["image_path", "category_id"],
        chunksize=chunk_size,
    )

    for i, categories_chunk in enumerate(category_iterator):
        chunk_num = start_chunk + i
        start_idx = processed_rows + i * chunk_size
        end_idx = min(start_idx + len(categories_chunk), len(all_attr_lines))

        attr_rows = []
        for line in all_attr_lines[start_idx:end_idx]:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                img_p, attr_labels = parts
                attr_indices = [
                    idx + 1 for idx, val in enumerate(attr_labels.split()) if val == "1"
                ]
                attr_rows.append([img_p, attr_indices])
            elif len(parts) == 1:
                attr_rows.append([parts[0], []])

        attributes_chunk = pd.DataFrame(attr_rows, columns=["image_path", "attr_labels"])
        chunk = categories_chunk.merge(attributes_chunk, on="image_path", how="left")
        del attributes_chunk

        chunk["category_name"] = chunk["category_id"].apply(
            lambda cid: cloth_categories.iloc[int(cid) - 1]["category"]
            if pd.notna(cid) and 0 <= int(cid) - 1 < len(cloth_categories)
            else None
        )
        chunk["type"] = chunk["category_id"].apply(
            lambda cid: map_garment_type(cid, cloth_categories)
        )
        chunk["season"] = chunk["attr_labels"].apply(
            lambda a: infer_from_mapping(a, season_mapping, "All")
        )
        chunk["occasion"] = chunk["attr_labels"].apply(
            lambda a: infer_from_mapping(a, occasion_mapping, "Casual")
        )
        chunk["style"] = chunk["attr_labels"].apply(
            lambda a: infer_from_mapping(a, style_mapping, "Casual")
        )
        chunk["name"] = chunk["image_path"].apply(lambda x: Path(str(x)).stem)
        chunk["color_palette"] = chunk["image_path"].apply(extract_color_palette)
        chunk["primary_color"] = chunk["color_palette"].apply(
            lambda x: x[0] if x and x[0] != "Unknown" else "Unknown"
        )

        chunk = chunk[
            ["name", "type", "primary_color", "color_palette", "style", "season", "occasion"]
        ].dropna(subset=["type"])

        chunk.to_csv(output_file, mode="a", header=first_chunk, index=False)
        first_chunk = False
        del chunk
        gc.collect()
        print(f"Chunk {chunk_num} completed.")

    print(f"Preprocessing complete. Saved to '{output_file}'.")
    preview_df = pd.read_csv(output_file, nrows=1000)
    print(preview_df.head())
    for col in ["type", "primary_color", "season", "occasion", "style"]:
        if col in preview_df.columns:
            print(preview_df.groupby(col).size())

Using local sample dataset at: /content/data/DeepFashion_sample
Sample preprocessing complete. Saved to 'data/DeepFashion_sample/processed_sample_with_palette.csv'.
        name      type primary_color               color_palette  style season occasion
    top_navy       Top       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual
   top_white       Top         White       [White, White, Beige] Casual    All   Casual
     top_red       Top         Brown       [Brown, Brown, Brown] Casual    All   Casual
bottom_denim    Bottom       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual
bottom_black    Bottom         Black     [Black, Black, Unknown] Casual    All   Casual
bottom_beige    Bottom        Silver    [Silver, Silver, Silver] Casual    All   Casual
 dress_olive Full-body       Unknown [Unknown, Unknown, Unknown] Casual    All   Casual


In [ ]:
import pandas as pd

# Load the full dataset
df = pd.read_csv('/content/drive/MyDrive/processed_deepfashion_with_palette.csv')
print(f"Total rows: {len(df)}")
print("\nFull dataset distributions:")
print(df.groupby('type').size())
print(df.groupby('primary_color').size())
print(df.groupby('season').size())
print(df.groupby('occasion').size())
print(df.groupby('style').size())

Total rows: 289222

Full dataset distributions:
type
Bottom     58723
Top       230499
dtype: int64
primary_color
Beige       69602
Black        3921
Brown         428
Coral         207
Gold           10
Gray        12007
Green           2
Indigo          3
Maroon         68
Navy           10
Olive           6
Orange         13
Pink         1101
Purple          4
Red            11
Silver      30426
Teal           75
Unknown     34596
White      136731
Yellow          1
dtype: int64
season
All       273899
Summer     15323
dtype: int64
occasion
Casual    286043
Party       2882
Sport        297
dtype: int64
style
Casual    288924
Sport        298
dtype: int64


In [ ]:
import ast
from pathlib import Path
import pandas as pd

sample_csv = Path("./data/DeepFashion_sample/processed_sample_with_palette.csv")
drive_csv = Path("/content/drive/MyDrive/processed_deepfashion_with_palette.csv")

if sample_csv.exists():
    csv_path = sample_csv
elif drive_csv.exists():
    csv_path = drive_csv
else:
    raise FileNotFoundError(
        f"No processed CSV found at '{sample_csv}' or '{drive_csv}'. "
        "Run the preprocessing cell first."
    )

df = pd.read_csv(csv_path)

# Parse stringified list back into a Python list if needed
if "color_palette" in df.columns and len(df) > 0 and isinstance(df["color_palette"].iloc[0], str):
    df["color_palette"] = df["color_palette"].apply(
        lambda val: ast.literal_eval(val) if isinstance(val, str) and val.startswith("[") else val
    )

print(f"Loaded from: {csv_path}")
print(f"Total rows: {len(df)}")
print("\nDataset distributions:")
for col in ["type", "primary_color", "season", "occasion", "style"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df.groupby(col, dropna=False).size())

Loaded from: data/DeepFashion_sample/processed_sample_with_palette.csv
Total rows: 7

Dataset distributions:

--- type ---
type
Bottom       3
Full-body    1
Top          3
dtype: int64

--- primary_color ---
primary_color
Black      1
Brown      1
Silver     1
Unknown    3
White      1
dtype: int64

--- season ---
season
All    7
dtype: int64

--- occasion ---
occasion
Casual    7
dtype: int64

--- style ---
style
Casual    7
dtype: int64


In [ ]:
import pandas as pd
import random
from google.colab import output

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/processed_deepfashion_with_palette.csv')

# Updated color compatibility matrix with weighted preferences
color_compatibility = {
    'White': {'White': 0.5, 'Beige': 0.9, 'Black': 1, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 1, 'Navy': 1.2, 'Green': 0.8, 'Teal': 0.8,
              'Red': 0.9, 'Maroon': 0.9, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Beige': {'White': 0.9, 'Beige': 0.5, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
              'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 1, 'Coral': 0.9, 'Olive': 1},
    'Black': {'White': 1, 'Beige': 0.9, 'Black': 0.5, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.9, 'Teal': 0.9,
              'Red': 0.9, 'Maroon': 0.9, 'Orange': 0.9, 'Yellow': 0.8, 'Gold': 1.2, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Gray': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.5, 'Silver': 0.9, 'Pink': 1, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
             'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 1.1, 'Indigo': 1.1, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Silver': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.5, 'Pink': 0.9, 'Blue': 1, 'Navy': 1, 'Green': 0.8, 'Teal': 0.8,
               'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 1.1, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.8},
    'Pink': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 1.2, 'Silver': 0.9, 'Pink': 0.5, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
             'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.8, 'Coral': 0.9, 'Olive': 0.8},
    'Blue': {'White': 1, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 1, 'Pink': 0.8, 'Blue': 0.5, 'Navy': 1.1, 'Green': 0.9, 'Teal': 0.9,
             'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.9},
    'Navy': {'White': 1, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 1, 'Pink': 0.8, 'Blue': 1.1, 'Navy': 0.5, 'Green': 0.9, 'Teal': 0.9,
             'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.9},
    'Green': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.5, 'Teal': 1.1,
              'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.7, 'Olive': 1},
    'Teal': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.9, 'Navy': 0.9, 'Green': 1.1, 'Teal': 0.5,
             'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.7, 'Olive': 1},
    'Red': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.6, 'Blue': 0.7, 'Navy': 0.7, 'Green': 0.6, 'Teal': 0.6,
            'Red': 0.5, 'Maroon': 1.1, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.7, 'Indigo': 0.7, 'Brown': 0.8, 'Coral': 0.8, 'Olive': 0.7},
    'Maroon': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.6, 'Blue': 0.7, 'Navy': 0.7, 'Green': 0.6, 'Teal': 0.6,
               'Red': 1.1, 'Maroon': 0.5, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.7, 'Indigo': 0.7, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.7},
    'Orange': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
               'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.5, 'Yellow': 1.1, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 0.9, 'Coral': 1, 'Olive': 0.8},
    'Yellow': {'White': 0.8, 'Beige': 0.8, 'Black': 0.8, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 1.1, 'Yellow': 0.5, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 0.8, 'Coral': 0.8, 'Olive': 0.7},
    'Gold': {'White': 0.9, 'Beige': 0.9, 'Black': 1, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
             'Red': 0.8, 'Maroon': 0.8, 'Orange': 1.1, 'Yellow': 1.1, 'Gold': 0.5, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Purple': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.5, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.8},
    'Indigo': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 1, 'Navy': 1.2, 'Green': 0.9, 'Teal': 0.9,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 1.2, 'Indigo': 0.5, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Brown': {'White': 0.9, 'Beige': 1, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.9, 'Teal': 0.9,
              'Red': 0.8, 'Maroon': 0.9, 'Orange': 0.9, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.5, 'Coral': 0.9, 'Olive': 1},
    'Coral': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
              'Red': 0.8, 'Maroon': 0.8, 'Orange': 1, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.5, 'Olive': 0.8},
    'Olive': {'White': 0.9, 'Beige': 1, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.8, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 1, 'Teal': 1,
              'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.9, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 1, 'Coral': 0.8, 'Olive': 0.5},
    'Unknown': {c: 1 for c in ['White', 'Beige', 'Black', 'Gray', 'Silver', 'Pink', 'Blue', 'Navy', 'Green', 'Teal',
                               'Red', 'Maroon', 'Orange', 'Yellow', 'Gold', 'Purple', 'Indigo', 'Brown', 'Coral', 'Olive', 'Unknown']}
}

def recommend_outfit(color=None, style=None, season=None, occasion=None):
    # Filter dataset based on user preferences
    filtered_df = df.copy()
    if style:
        filtered_df = filtered_df[filtered_df['style'] == style]
    if season:
        filtered_df = filtered_df[filtered_df['season'].isin([season, 'All'])]
    if occasion:
        filtered_df = filtered_df[filtered_df['occasion'] == occasion]

    # Ensure we have items for Top and Bottom
    if len(filtered_df) == 0:
        print("No items match your preferences. Relaxing constraints...")
        filtered_df = df.copy()
        if season:
            filtered_df = filtered_df[filtered_df['season'].isin([season, 'All'])]

    # Select Top with specified color (if provided)
    top_items = filtered_df[filtered_df['type'] == 'Top']
    if color:
        top_items = top_items[top_items['primary_color'] == color]
    bottom_items = filtered_df[filtered_df['type'] == 'Bottom']  # No color filter for bottoms

    # Debug: Print the number of items after filtering
    print(f"After filtering: {len(top_items)} Tops, {len(bottom_items)} Bottoms")
    print(f"Top colors: {top_items['primary_color'].value_counts().to_dict()}")
    print(f"Bottom colors: {bottom_items['primary_color'].value_counts().to_dict()}")

    if len(top_items) == 0 or len(bottom_items) == 0:
        raise ValueError("No items available for the given filters.")

    top = top_items.sample(1).iloc[0]
    best_bottom = None
    max_score = 0

    # First, try to find a compatible bottom with a different color
    for _, bottom in bottom_items.iterrows():
        if bottom['primary_color'] == top['primary_color']:
            continue  # Skip same-color items
        score = color_compatibility[top['primary_color']].get(bottom['primary_color'], 0)
        # Apply season and occasion penalties
        if bottom['season'] != top['season'] and bottom['season'] != 'All':
            score *= 0.8
        if bottom['occasion'] != top['occasion']:
            score *= 0.8
        if score > max_score:
            max_score = score
            best_bottom = bottom

    # Check if best_bottom is not None and score meets threshold (0.5)
    if best_bottom is not None and max_score >= 0.5:
        outfit = {'Top': top.to_dict(), 'Bottom': best_bottom.to_dict()}
    else:
        print("No compatible Bottom with different color found. Trying any compatible Bottom...")
        # Try again without the different-color restriction
        for _, bottom in bottom_items.iterrows():
            score = color_compatibility[top['primary_color']].get(bottom['primary_color'], 0)
            if bottom['season'] != top['season'] and bottom['season'] != 'All':
                score *= 0.8
            if bottom['occasion'] != top['occasion']:
                score *= 0.8
            if score > max_score:
                max_score = score
                best_bottom = bottom

        if best_bottom is not None and max_score >= 0.5:
            outfit = {'Top': top.to_dict(), 'Bottom': best_bottom.to_dict()}
        else:
            print("No compatible Bottom found at all. Using random match with different color...")
            outfit = {'Top': top.to_dict()}
            # Use full dataset for bottoms to find a different color
            different_color_bottoms = df[(df['type'] == 'Bottom') & (df['primary_color'] != top['primary_color'])]
            if len(different_color_bottoms) > 0:
                outfit['Bottom'] = different_color_bottoms.sample(1).iloc[0].to_dict()
            else:
                print("No different color Bottoms available. Using same color as last resort...")
                outfit['Bottom'] = bottom_items.sample(1).iloc[0].to_dict()

    # Display the outfit
    print("\nRecommended Outfit:")
    for item_type, item in outfit.items():
        print(f"{item_type}: {item['name']} (Color: {item['primary_color']}, Style: {item['style']}, Season: {item['season']}, Occasion: {item['occasion']})")

# Interactive input form
color = input("Enter preferred color (e.g., White, Black, leave blank for any): ") or None
style = input("Enter preferred style (e.g., Casual, Sport, leave blank for any): ") or None
season = input("Enter preferred season (e.g., Summer, All, leave blank for any): ") or None
occasion = input("Enter preferred occasion (e.g., Casual, Party, leave blank for any): ") or None
recommend_outfit(color=color, style=style, season=season, occasion=occasion)*

Enter preferred color (e.g., White, Black, leave blank for any): Navy
Enter preferred style (e.g., Casual, Sport, leave blank for any): Casual
Enter preferred season (e.g., Summer, All, leave blank for any): Summer
Enter preferred occasion (e.g., Casual, Party, leave blank for any): Casual
After filtering: 9 Tops, 58435 Bottoms
Top colors: {'Navy': 9}
Bottom colors: {'White': 27043, 'Beige': 12965, 'Unknown': 8195, 'Silver': 6247, 'Gray': 2891, 'Black': 699, 'Pink': 246, 'Brown': 70, 'Coral': 49, 'Teal': 14, 'Maroon': 8, 'Orange': 2, 'Red': 2, 'Olive': 1, 'Navy': 1, 'Green': 1, 'Gold': 1}

Recommended Outfit:
Top: img_00000017 (Color: Navy, Style: Casual, Season: All, Occasion: Casual)
Bottom: img_00000003 (Color: White, Style: Casual, Season: All, Occasion: Casual)


In [ ]:
import random
from pathlib import Path
import pandas as pd

try:
    from google.colab import output  # noqa: F401
except ImportError:
    pass

# Load processed dataset (sample first, Google Drive fallback)
sample_csv = Path("./data/DeepFashion_sample/processed_sample_with_palette.csv")
drive_csv = Path("/content/drive/MyDrive/processed_deepfashion_with_palette.csv")

if sample_csv.exists():
    csv_path = sample_csv
elif drive_csv.exists():
    csv_path = drive_csv
else:
    raise FileNotFoundError(
        f"Processed CSV not found at '{sample_csv}' or '{drive_csv}'. Run the preprocessing cell first."
    )

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} items from {csv_path}")

# Updated color compatibility matrix with weighted preferences
color_compatibility = {
    'White': {'White': 0.5, 'Beige': 0.9, 'Black': 1, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 1, 'Navy': 1.2, 'Green': 0.8, 'Teal': 0.8,
              'Red': 0.9, 'Maroon': 0.9, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Beige': {'White': 0.9, 'Beige': 0.5, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
              'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 1, 'Coral': 0.9, 'Olive': 1},
    'Black': {'White': 1, 'Beige': 0.9, 'Black': 0.5, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.9, 'Teal': 0.9,
              'Red': 0.9, 'Maroon': 0.9, 'Orange': 0.9, 'Yellow': 0.8, 'Gold': 1.2, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Gray': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.5, 'Silver': 0.9, 'Pink': 1, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
             'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 1.1, 'Indigo': 1.1, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Silver': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.5, 'Pink': 0.9, 'Blue': 1, 'Navy': 1, 'Green': 0.8, 'Teal': 0.8,
               'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 1.1, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.8},
    'Pink': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 1.2, 'Silver': 0.9, 'Pink': 0.5, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
             'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.8, 'Coral': 0.9, 'Olive': 0.8},
    'Blue': {'White': 1, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 1, 'Pink': 0.8, 'Blue': 0.5, 'Navy': 1.1, 'Green': 0.9, 'Teal': 0.9,
             'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.9},
    'Navy': {'White': 1, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 1, 'Pink': 0.8, 'Blue': 1.1, 'Navy': 0.5, 'Green': 0.9, 'Teal': 0.9,
             'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.9},
    'Green': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.5, 'Teal': 1.1,
              'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.7, 'Olive': 1},
    'Teal': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.9, 'Navy': 0.9, 'Green': 1.1, 'Teal': 0.5,
             'Red': 0.6, 'Maroon': 0.6, 'Orange': 0.7, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.7, 'Olive': 1},
    'Red': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.6, 'Blue': 0.7, 'Navy': 0.7, 'Green': 0.6, 'Teal': 0.6,
            'Red': 0.5, 'Maroon': 1.1, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.7, 'Indigo': 0.7, 'Brown': 0.8, 'Coral': 0.8, 'Olive': 0.7},
    'Maroon': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.6, 'Blue': 0.7, 'Navy': 0.7, 'Green': 0.6, 'Teal': 0.6,
               'Red': 1.1, 'Maroon': 0.5, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.8, 'Purple': 0.7, 'Indigo': 0.7, 'Brown': 0.9, 'Coral': 0.8, 'Olive': 0.7},
    'Orange': {'White': 0.8, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
               'Red': 0.8, 'Maroon': 0.8, 'Orange': 0.5, 'Yellow': 1.1, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 0.9, 'Coral': 1, 'Olive': 0.8},
    'Yellow': {'White': 0.8, 'Beige': 0.8, 'Black': 0.8, 'Gray': 0.8, 'Silver': 0.8, 'Pink': 0.7, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 1.1, 'Yellow': 0.5, 'Gold': 1.1, 'Purple': 0.8, 'Indigo': 0.8, 'Brown': 0.8, 'Coral': 0.8, 'Olive': 0.7},
    'Gold': {'White': 0.9, 'Beige': 0.9, 'Black': 1, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
             'Red': 0.8, 'Maroon': 0.8, 'Orange': 1.1, 'Yellow': 1.1, 'Gold': 0.5, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Purple': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.8, 'Teal': 0.8,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.5, 'Indigo': 1.2, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.8},
    'Indigo': {'White': 0.9, 'Beige': 0.8, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 1, 'Navy': 1.2, 'Green': 0.9, 'Teal': 0.9,
               'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 1.2, 'Indigo': 0.5, 'Brown': 0.9, 'Coral': 0.9, 'Olive': 0.9},
    'Brown': {'White': 0.9, 'Beige': 1, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 0.9, 'Teal': 0.9,
              'Red': 0.8, 'Maroon': 0.9, 'Orange': 0.9, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.5, 'Coral': 0.9, 'Olive': 1},
    'Coral': {'White': 0.9, 'Beige': 0.9, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.9, 'Pink': 0.9, 'Blue': 0.8, 'Navy': 0.8, 'Green': 0.7, 'Teal': 0.7,
              'Red': 0.8, 'Maroon': 0.8, 'Orange': 1, 'Yellow': 0.8, 'Gold': 0.9, 'Purple': 0.9, 'Indigo': 0.9, 'Brown': 0.9, 'Coral': 0.5, 'Olive': 0.8},
    'Olive': {'White': 0.9, 'Beige': 1, 'Black': 0.9, 'Gray': 0.9, 'Silver': 0.8, 'Pink': 0.8, 'Blue': 0.9, 'Navy': 0.9, 'Green': 1, 'Teal': 1,
              'Red': 0.7, 'Maroon': 0.7, 'Orange': 0.8, 'Yellow': 0.7, 'Gold': 0.9, 'Purple': 0.8, 'Indigo': 0.9, 'Brown': 1, 'Coral': 0.8, 'Olive': 0.5},
    'Unknown': {c: 1 for c in ['White', 'Beige', 'Black', 'Gray', 'Silver', 'Pink', 'Blue', 'Navy', 'Green', 'Teal',
                               'Red', 'Maroon', 'Orange', 'Yellow', 'Gold', 'Purple', 'Indigo', 'Brown', 'Coral', 'Olive', 'Unknown']}
}


def get_compat_score(top_color, bottom_color):
    row_scores = color_compatibility.get(top_color, color_compatibility["Unknown"])
    return row_scores.get(bottom_color, 0.5)


def recommend_outfit(color=None, style=None, season=None, occasion=None):
    # Filter dataset based on user preferences
    filtered_df = df.copy()
    if style:
        filtered_df = filtered_df[filtered_df['style'] == style]
    if season:
        filtered_df = filtered_df[filtered_df['season'].isin([season, 'All'])]
    if occasion:
        filtered_df = filtered_df[filtered_df['occasion'] == occasion]

    # Relax constraints if no items match
    if len(filtered_df) == 0:
        print("No items match your preferences. Relaxing constraints...")
        filtered_df = df.copy()
        if season:
            filtered_df = filtered_df[filtered_df['season'].isin([season, 'All'])]

    top_items = filtered_df[filtered_df['type'] == 'Top']
    if color:
        top_items = top_items[top_items['primary_color'] == color]
    bottom_items = filtered_df[filtered_df['type'] == 'Bottom']

    print(f"After filtering: {len(top_items)} Tops, {len(bottom_items)} Bottoms")
    print(f"Top colors: {top_items['primary_color'].value_counts().to_dict()}")
    print(f"Bottom colors: {bottom_items['primary_color'].value_counts().to_dict()}")

    if len(top_items) == 0 or len(bottom_items) == 0:
        raise ValueError("No items available for the given filters.")

    top = top_items.sample(1).iloc[0]
    best_bottom = None
    max_score = 0

    # 1. Try compatible bottom with a different color
    for _, bottom in bottom_items.iterrows():
        if bottom['primary_color'] == top['primary_color']:
            continue
        score = get_compat_score(top['primary_color'], bottom['primary_color'])
        if bottom['season'] != top['season'] and bottom['season'] != 'All':
            score *= 0.8
        if bottom['occasion'] != top['occasion']:
            score *= 0.8
        if score > max_score:
            max_score = score
            best_bottom = bottom

    # 2. Fallback to any compatible bottom (including same color)
    if best_bottom is not None and max_score >= 0.5:
        outfit = {'Top': top.to_dict(), 'Bottom': best_bottom.to_dict()}
    else:
        print("No compatible Bottom with different color found. Trying any compatible Bottom...")
        for _, bottom in bottom_items.iterrows():
            score = get_compat_score(top['primary_color'], bottom['primary_color'])
            if bottom['season'] != top['season'] and bottom['season'] != 'All':
                score *= 0.8
            if bottom['occasion'] != top['occasion']:
                score *= 0.8
            if score > max_score:
                max_score = score
                best_bottom = bottom

        if best_bottom is not None and max_score >= 0.5:
            outfit = {'Top': top.to_dict(), 'Bottom': best_bottom.to_dict()}
        else:
            print("No compatible Bottom found at all. Using random match with different color...")
            outfit = {'Top': top.to_dict()}
            different_color_bottoms = df[(df['type'] == 'Bottom') & (df['primary_color'] != top['primary_color'])]
            if len(different_color_bottoms) > 0:
                outfit['Bottom'] = different_color_bottoms.sample(1).iloc[0].to_dict()
            else:
                print("No different color Bottoms available. Using same color as last resort...")
                outfit['Bottom'] = bottom_items.sample(1).iloc[0].to_dict()

    print("\nRecommended Outfit:")
    for item_type, item in outfit.items():
        print(f"{item_type}: {item['name']} (Color: {item['primary_color']}, Style: {item['style']}, Season: {item['season']}, Occasion: {item['occasion']})")


# Interactive input form (press Enter to leave any filter blank)
color = input("Enter preferred color (e.g., White, Black, leave blank for any): ").strip() or None
style = input("Enter preferred style (e.g., Casual, Sport, leave blank for any): ").strip() or None
season = input("Enter preferred season (e.g., Summer, All, leave blank for any): ").strip() or None
occasion = input("Enter preferred occasion (e.g., Casual, Party, leave blank for any): ").strip() or None
recommend_outfit(color=color, style=style, season=season, occasion=occasion)

Loaded 7 items from data/DeepFashion_sample/processed_sample_with_palette.csv
Enter preferred color (e.g., White, Black, leave blank for any): White
Enter preferred style (e.g., Casual, Sport, leave blank for any): Casual
Enter preferred season (e.g., Summer, All, leave blank for any): 
Enter preferred occasion (e.g., Casual, Party, leave blank for any): 
After filtering: 1 Tops, 3 Bottoms
Top colors: {'White': 1}
Bottom colors: {'Unknown': 1, 'Black': 1, 'Silver': 1}

Recommended Outfit:
Top: top_white (Color: White, Style: Casual, Season: All, Occasion: Casual)
Bottom: bottom_black (Color: Black, Style: Casual, Season: All, Occasion: Casual)


In [ ]:
import os
import pandas as pd
output_file = '/content/drive/MyDrive/processed_deepfashion_with_palette.csv'
print(f"File exists: {os.path.exists(output_file)}, Size: {os.path.getsize(output_file)} bytes")
df = pd.read_csv(output_file)
processed_rows = len(df)
print(f"Processed rows: {processed_rows}")

File exists: True, Size: 13018121 bytes
Processed rows: 174000


In [ ]:
processed_df = pd.read_csv('/content/drive/MyDrive/processed_deepfashion_coarse.csv', nrows=1000)
print(processed_df.head())
print(processed_df.groupby('type').size())  # Check distribution of Top, Bottom, Footwear

           name type    color   style  season occasion
0  img_00000001  Top  Unknown  Casual     All   Casual
1  img_00000002  Top  Unknown  Casual     All   Casual
2  img_00000003  Top  Unknown  Casual  Summer   Casual
3  img_00000004  Top  Unknown  Casual     All   Casual
4  img_00000005  Top  Unknown  Casual     All   Casual
type
Top    1000
dtype: int64


In [ ]:
# preprocess_deepfashion.py
import pandas as pd
import os
import gc
import re

# Set dataset path in Google Drive
dataset_path = '/content/drive/MyDrive/DeepFashion'
anno_path = os.path.join(dataset_path, 'Anno_coarse')

# File paths for DeepFashion annotations
category_file = os.path.join(anno_path, 'list_category_img.txt')
cloth_category_file = os.path.join(anno_path, 'list_category_cloth.txt')
attribute_file = os.path.join(anno_path, 'list_attr_img.txt')
attribute_list_file = os.path.join(anno_path, 'list_attr_cloth.txt')

# Check if files exist
for file_path in [category_file, cloth_category_file, attribute_file, attribute_list_file]:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Required file not found: {file_path}")

# Load category mappings
cloth_categories = pd.read_csv(cloth_category_file, skiprows=2, sep='\s+', names=['category', 'type'])

# Map categories to Top, Bottom, Footwear
category_to_type = {
    'T-shirt': 'Top', 'Shirt': 'Top', 'Sweater': 'Top', 'Jacket': 'Top', 'Blouse': 'Top',
    'Tank': 'Top', 'Hoodie': 'Top', 'Cardigan': 'Top', 'Pants': 'Bottom', 'Jeans': 'Bottom',
    'Shorts': 'Bottom', 'Skirt': 'Bottom', 'Leggings': 'Bottom', 'Trousers': 'Bottom',
    'Sneakers': 'Footwear', 'Boots': 'Footwear', 'Sandals': 'Footwear', 'Shoes': 'Footwear',
    'Heels': 'Footwear', 'Flip-Flops': 'Footwear'
}
type_mapping = {1: 'Top', 2: 'Bottom', 3: 'Top'}

# Load attribute list
def parse_attribute_file(file_path):
    data = []
    with open(file_path, 'r') as f:
        next(f)  # Skip "1000"
        next(f)  # Skip "attribute_name  attribute_type"
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                attribute = ' '.join(parts[:-1])
                try:
                    type_value = int(parts[-1])
                    data.append([attribute, type_value])
                except ValueError:
                    print(f"Skipping line with non-numeric type: {line.strip()}")
                    continue
    return pd.DataFrame(data, columns=['attribute', 'type'])

attributes_list = parse_attribute_file(attribute_list_file)
print("First 50 rows of attributes_list:")
print(attributes_list.head(50))  # Print more rows to look for colors

# Create a mapping of attribute indices to names
attr_index_to_name = {i + 1: attr for i, attr in enumerate(attributes_list['attribute'])}

# Inference mappings
season_mapping = {
    'short sleeves': 'Summer', 'sleeveless': 'Summer', 'long sleeves': 'Winter',
    'wool': 'Winter', 'cotton': 'Summer', 'light fabric': 'Spring', 'thick fabric': 'Autumn'
}
occasion_mapping = {
    'formal': 'Formal', 'casual': 'Casual', 'sporty': 'Sport', 'party': 'Party', 'business': 'Work'
}
style_mapping = {
    'casual': 'Casual', 'formal': 'Formal', 'sporty': 'Sport', 'street': 'Street'
}

# Expanded list of known colors
known_colors = [
    'blue', 'black', 'white', 'red', 'green', 'gray', 'yellow', 'pink', 'purple', 'orange',
    'brown', 'beige', 'gold', 'silver', 'navy', 'teal', 'maroon', 'olive', 'coral', 'indigo'
]

def infer_season(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, season in season_mapping.items():
            if key in attr.lower():
                return season
    return 'All'

def infer_occasion(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, occasion in occasion_mapping.items():
            if key in attr.lower():
                return occasion
    return 'Casual'

def infer_style(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '')
        for key, style in style_mapping.items():
            if key in attr.lower():
                return style
    return 'Casual'

def extract_color(attrs):
    for attr_idx in attrs:
        attr = attr_index_to_name.get(int(attr_idx), '').lower().strip()
        for color in known_colors:
            if re.search(r'\b' + re.escape(color) + r'\b|\b' + re.escape(color) + r'[\s-]', attr):
                print(f"Matched color {color} from attribute: {attr_index_to_name.get(int(attr_idx), '')}")
                print(f"Full attribute context for {attr_index_to_name.get(int(attr_idx), '')}: {', '.join([attr_index_to_name.get(int(idx), 'Unknown') for idx in attrs if pd.notna(idx)])}")
                return color.capitalize()
    return 'Unknown'

# Process data in chunks
chunk_size = 10000
output_file = '/content/drive/MyDrive/processed_deepfashion_coarse.csv'
first_chunk = True

category_iterator = pd.read_csv(category_file, skiprows=2, sep='\s+', names=['image_path', 'category_id'], chunksize=chunk_size)
for i, categories_chunk in enumerate(category_iterator):
    print(f"Processing chunk {i + 1}...")
    # Load corresponding attributes chunk
    attr_data = []
    with open(attribute_file, 'r') as f:
        lines = f.readlines()[2:]
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        for line in lines[start_idx:end_idx]:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                image_path, attr_labels = parts
                attr_values = attr_labels.split()
                attr_indices = [idx + 1 for idx, val in enumerate(attr_values) if val == '1']
                attr_data.append([image_path, attr_indices])
                if i == 0 and len(attr_data) <= 5:
                    attr_names = [attr_index_to_name.get(idx, f"Unknown_{idx}") for idx in attr_indices]
                    print(f"Image: {image_path}, Attributes: {attr_names}")
            else:
                attr_data.append([parts[0], []])
    attributes_chunk = pd.DataFrame(attr_data, columns=['image_path', 'attr_labels'])

    # Merge chunks
    chunk = categories_chunk.merge(attributes_chunk, on='image_path')
    del attributes_chunk
    gc.collect()

    # Map categories
    chunk['category_name'] = chunk['category_id'].map(cloth_categories['category'])
    chunk['type'] = chunk['category_name'].map(category_to_type).fillna(
        chunk['category_id'].map(cloth_categories['type']).map(type_mapping)
    )

    # Apply inference
    chunk['color'] = chunk['attr_labels'].apply(extract_color)
    chunk['season'] = chunk['attr_labels'].apply(infer_season)
    chunk['occasion'] = chunk['attr_labels'].apply(infer_occasion)
    chunk['style'] = chunk['attr_labels'].apply(infer_style)
    chunk['name'] = chunk['image_path'].apply(lambda x: x.split('/')[-1].replace('.jpg', ''))

    # Select columns and filter
    chunk = chunk[['name', 'type', 'color', 'style', 'season', 'occasion']]
    chunk = chunk[chunk['type'].notnull()]

    # Save to file
    mode = 'w' if first_chunk else 'a'
    header = first_chunk
    chunk.to_csv(output_file, mode=mode, header=header, index=False)
    first_chunk = False

    del chunk
    gc.collect()

print(f"Preprocessing complete. Saved to '{output_file}'.")

# Load a small sample for verification
processed_df = pd.read_csv(output_file, nrows=1000)
print(processed_df.head())
print(processed_df.groupby('type').size())
print(processed_df.groupby('color').size())
print(processed_df.groupby('season').size())
print(processed_df.groupby('occasion').size())
print(processed_df.groupby('style').size())

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
Matched color red from attribute: red
Full attribute context for red: applique, floral, long sleeve, red, rose, sheer, sleeve
Matched color pink from attribute: pink
Full attribute context for pink: pink, rose, soft
Matched color pink from attribute: pink
Full attribute context for pink: chiffon, pink, rose
Matched color pink from attribute: pink
Full attribute context for pink: floral, pink, rose, summer
Matched color red from attribute: red
Full attribute context for red: floral, red
Matched color pink from attribute: pink
Full attribute context for pink: pink, rose
Matched color pink from attribute: pink
Full attribute context for pink: organza, pink, rose, sweetheart
Matched color pink from attribute: pink
Full attribute context for pink: lace, pink, rose
Matched color red from attribute: red
Full attribute context for red: bodycon, knit, long sleeve, neck ribbed, red, ribbed, scoop, sleeve
Matched color 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Install dependencies if needed
!pip install pandas scikit-learn numpy

# Color compatibility matrix
color_compatibility = {
    'Blue': {'Blue': 1, 'Black': 1, 'White': 1, 'Red': 0.5, 'Green': 1, 'Gray': 1},
    'Black': {'Blue': 1, 'Black': 1, 'White': 1, 'Red': 1, 'Green': 1, 'Gray': 1},
    'White': {'Blue': 1, 'Black': 1, 'White': 1, 'Red': 1, 'Green': 1, 'Gray': 1},
    'Red': {'Blue': 0.5, 'Black': 1, 'White': 1, 'Red': 1, 'Green': 0.5, 'Gray': 1},
    'Green': {'Blue': 1, 'Black': 1, 'White': 1, 'Red': 0.5, 'Green': 1, 'Gray': 1},
    'Gray': {'Blue': 1, 'Black': 1, 'White': 1, 'Red': 1, 'Green': 1, 'Gray': 1}
}

def load_data(file_path='/content/drive/MyDrive/processed_deepfashion_coarse.csv'):
    # Skip rows to start just before the first Bottom item (row 68,410)
    # Adjust for header row: skiprows=68000 means we start at row 68,001 (0-based index)
    df = pd.read_csv(file_path, skiprows=68000, nrows=5000)
    # Since skiprows skips the header, manually set column names
    df.columns = ['name', 'type', 'color', 'style', 'season', 'occasion']
    return df

def encode_features(df):
    categorical_features = ['type', 'color', 'style', 'season', 'occasion']
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_features = encoder.fit_transform(df[categorical_features])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_features))
    encoded_df['name'] = df['name']
    return encoded_df, encoder

def compute_color_score(item1_idx, item2_idx, df):
    color1 = df.loc[item1_idx, 'color']
    color2 = df.loc[item2_idx, 'color']
    return color_compatibility.get(color1, {}).get(color2, 0.5)

def recommend_outfit(df, style_preference='Casual', season_preference='All', occasion_preference='Casual'):
    encoded_df, encoder = encode_features(df)
    filtered_df = encoded_df[encoded_df[f'style_{style_preference}'] == 1]
    if season_preference != 'All':
        filtered_df = filtered_df[filtered_df[f'season_{season_preference}'] == 1]
    filtered_df = filtered_df[filtered_df[f'occasion_{occasion_preference}'] == 1]

    tops = filtered_df[filtered_df['type_Top'] == 1]
    bottoms = filtered_df[filtered_df['type_Bottom'] == 1]

    if tops.empty:
        return "No Top items found for the given preferences."
    if bottoms.empty:
        return "No Bottom items found for the given preferences."

    features = encoded_df.drop(columns=['name']).values
    similarity_matrix = cosine_similarity(features)

    best_score = -1
    best_outfit = None

    tops = tops.head(10)  # Limit to 10 items to reduce computation
    bottoms = bottoms.head(10)

    for top_idx in tops.index:
        for bottom_idx in bottoms.index:
            sim_score = similarity_matrix[top_idx, bottom_idx]
            color_score = compute_color_score(top_idx, bottom_idx, df)
            total_score = 0.7 * sim_score + 0.3 * color_score

            if total_score > best_score:
                best_score = total_score
                best_outfit = {
                    'Top': df.loc[top_idx, 'name'],
                    'Bottom': df.loc[bottom_idx, 'name']
                }

    return best_outfit

# Example usage
if __name__ == '__main__':
    df = load_data()
    print("Loaded data shape:", df.shape)
    print(df.groupby('type').size())
    print(df.head())  # Inspect a sample of the data

    outfit = recommend_outfit(df, style_preference='Casual', season_preference='All', occasion_preference='Casual')
    print('Suggested Outfit:', outfit)

Loaded data shape: (5000, 6)
type
Bottom      58
Top       4942
dtype: int64
           name type    color   style season occasion
0  img_00000045  Top  Unknown  Casual    All   Casual
1  img_00000046  Top  Unknown  Casual    All   Casual
2  img_00000047  Top  Unknown  Casual    All   Casual
3  img_00000048  Top  Unknown  Casual    All   Casual
4  img_00000049  Top  Unknown  Casual    All   Casual
Suggested Outfit: {'Top': 'img_00000045', 'Bottom': 'img_00000001'}
